In [1]:
import eu9_osm
import geopandas as gpd
import matplotlib.pyplot as plt
import h3pandas
from shapely.affinity import rotate

In [2]:
# Params
lat = 38.7115
lon = -9.1300

distance = 800
resolution = 8

crs = "EPSG:3763"

In [3]:
# OSM data
tags = {
    "building": True,

    "highway": [
        "primary", "secondary", "tertiary",
        "residential", "unclassified", "service",
        "pedestrian", "footway", "cycleway",
        "living_street", "path", "steps"
    ],

    "leisure": ["park", "garden"],
    "landuse": ["grass", "forest", "recreation_ground"],
    "natural": ["wood", "scrub", "grassland"]
}

osm = eu9_osm.features_from_point(location=(lat, lon), tags=tags, dist=distance, shape="circle") 
osm = osm[osm.geometry.geom_type != "Point"]
osm = osm.to_crs(crs)

In [4]:
# New OSM gdfs
buildings = osm[osm["building"].notna() & osm.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

roads = osm[osm["highway"].notna() & osm.geom_type.isin(["LineString", "MultiLineString"])].copy()

greens = osm[
    (
        osm["leisure"].notna() |
        osm["landuse"].isin(["grass", "forest", "recreation_ground"]) |
        osm["natural"].isin(["wood", "scrub", "grassland"])
    ) & osm.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

In [5]:
# Hexagon
point = gpd.GeoDataFrame(geometry=gpd.points_from_xy([lon], [lat]), crs="EPSG:4326")

hexagon = (
    point.h3
    .geo_to_h3(resolution)
    .h3.h3_to_geo_boundary()
    .to_crs(crs)
)

# Inner hexagon 
inner_hex = hexagon.copy()
inner_hex.geometry = inner_hex.geometry.scale(xfact=0.95, yfact=0.95,  origin="centroid")

In [6]:
# Clip to inner hexagon
buildings = gpd.clip(buildings, inner_hex)
roads = gpd.clip(roads, inner_hex)
greens = gpd.clip(greens, inner_hex)

In [7]:
# Rotate
angle = 5.25 
origin = hexagon.geometry.iloc[0].centroid
hexagon["geometry"] = hexagon.geometry.apply(lambda g: rotate(g, angle, origin=origin))
inner_hex["geometry"] = inner_hex.geometry.apply(lambda g: rotate(g, angle, origin=origin))
buildings["geometry"] = buildings.geometry.apply(lambda g: rotate(g, angle, origin=origin))
roads["geometry"] = roads.geometry.apply(lambda g: rotate(g, angle, origin=origin))
greens["geometry"] = greens.geometry.apply(lambda g: rotate(g, angle, origin=origin))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10), facecolor="#252D35")

hexagon.plot(ax=ax, color="#252D35", ec="none", zorder=1)
greens.plot(ax=ax, color="#6FA36B", ec="none", zorder=2)
buildings.plot(ax=ax, color="#E8D9B5", ec="none", zorder=3)
roads.plot(ax=ax, color="#F6A04D", lw=0.5, zorder=4)
hexagon.boundary.plot(ax=ax, color="#FFF0CC", lw=2.5, zorder=5)

ax.set_facecolor("#20272B")
ax.axis("off")

fig.text(0.5, 0.975,
         "One hexagon of Lisbon",
         color="#FFF0CC", size=22, weight="bold", ha="center", va="top")

fig.text(0.5, 0.05,
         "Data: OpenStreetMap, licensed under ODbL | Map: Bernadett Piros",  
         color="#FFF0CC", size=8, ha="center")

plt.tight_layout()
#plt.show()

fig.savefig("hexagon.png", bbox_inches="tight", pad_inches=0.2) 